In [ ]:
import os
import pandas as pd 
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, confusion_matrix, classification_report
import xgboost as xgb

RS = 42

In [ ]:
data = pd.read_csv("../../dados/features_creditcard.csv")

In [ ]:
data['Class'].value_counts()

In [ ]:
from imblearn.over_sampling import SMOTE
from collections import Counter

X = data.drop('Class', axis=1)
y = data['Class']

X_train, X_test, y_train, y_test = train_test_split(
    X, 
    y, 
    test_size=0.2, 
    random_state=RS, 
    stratify=y 
)

smote = SMOTE(random_state=RS)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

print(f"Distribuição após o SMOTE no treino: {Counter(y_train_resampled)}")
print(f"Distribuição no teste: {Counter(y_test)}")

In [ ]:
from xgboost import XGBClassifier
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.model_selection import RandomizedSearchCV
import scipy.stats as stats
import random

xgb_model = XGBClassifier(
    objective='binary:logistic',
    eval_metric='auc',
    random_state=42,
    verbosity=0 
)

pipeline = ImbPipeline([
    ('smote', SMOTE(random_state=42)),
    ('model', xgb_model)
])

param_dist = {
    'model__learning_rate': stats.uniform(0.01, 0.1),
    'model__max_depth': [3, 5, 7, 9, 12],
    'model__subsample': [0.6, 0.8, 0.9, 1.0],
    'model__colsample_bytree': [0.7, 0.8, 0.9, 1.0],
    'smote__k_neighbors': [3, 5, 7, 9, 11],
}

random_search = RandomizedSearchCV(
    estimator=pipeline,
    param_distributions=param_dist,
    n_iter=100,          
    cv=5,               
    scoring='roc_auc',  
    random_state=42,
    n_jobs=-1,
    verbose=1
)

random_search.fit(X_train, y_train)

In [ ]:
from sklearn.metrics import recall_score

model = random_search.best_estimator_
y_pred_proba = model.predict_proba(X_test)[:, 1]
auc_score = roc_auc_score(y_test, y_pred_proba)
recall = recall_score(y_test, model.predict(X_test))
print(f"AUC-ROC Score: {auc_score:.4f}")
print(f"Recall: {recall:.4f}")


In [ ]:
conf_matrix = confusion_matrix(y_test, model.predict(X_test), normalize='true')
sns.heatmap(conf_matrix, annot=True, cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()

In [ ]:
import joblib

artifact = {
    "model": random_search.best_estimator_,
    "mean_amount": float(X_train["Amount"].mean()),
}
joblib.dump(artifact, "../../dados/model_fraud.pkl")
print(f"Modelo salvo em dados/model_fraud.pkl")
print(f"mean_amount usado no treino: {artifact['mean_amount']:.4f}")